# 01 — OULAD DuckDB Ingestion, 8 SQL Data Quality Checks & Returning/New Overlap (`P1-B1`)

This notebook verifies the **real OULAD dataset** (UCI ID 349, CC BY 4.0):
1. Loads all 7 raw CSVs into `data/oulad.duckdb` using `read_csv_auto(..., nullstr='?')`.
2. Runs the 8 mandatory SQL Data Quality Checks and logs all real-world anomalies transparently (never deleting checks that uncover real anomalies such as check #7).
3. Profiles `imd_band` missing rate and `date_registration` (`NULL` and `>= 14/21/28`).
4. Computes `Returning vs New Students` on `2014B` and `2014J` against Train (`2013B + 2013J`).
5. Saves structured real-data evidence to `reports/data_quality.json`.

In [ ]:
import json
from datetime import datetime, timezone

import pandas as pd

from oulad.config import DB_PATH, RAW_DIR, REPORTS_DIR
from oulad.db import (
    compute_student_overlap,
    get_connection,
    get_registration_and_missing_stats,
    load_raw_tables,
    run_quality_checks,
)

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
con = get_connection(DB_PATH)
row_counts = load_raw_tables(RAW_DIR, con)
pd.Series(row_counts, name='row_count').to_frame()

In [ ]:
quality_checks = run_quality_checks(con)
reg_stats = get_registration_and_missing_stats(con)
overlap = compute_student_overlap(con)

print('=== 8 SQL Quality Checks (Real OULAD DB) ===')
display(pd.Series(quality_checks, name='n_violations').to_frame())

print('\n=== Registration & Missing Stats ===')
display(pd.Series(reg_stats, name='value').to_frame())

print('\n=== Returning vs New Students (vs Train 2013B + 2013J) ===')
display(pd.DataFrame(overlap).T)

In [ ]:
report = {
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'dataset': 'OULAD (UCI ID 349, CC BY 4.0)',
    'row_counts': row_counts,
    'quality_checks': quality_checks,
    'registration_and_missing_stats': reg_stats,
    'returning_vs_new_students': overlap,
}
out_path = REPORTS_DIR / 'data_quality.json'
out_path.write_text(json.dumps(report, indent=2), encoding='utf-8')
con.close()
print(f'Saved real-data quality report to: {out_path}')